# BIKV v12 Stage F2: medium-context preregistered comparison
The run first qualifies the full-context backbone and task oracle, then evaluates frozen selectors at an approximately 4k-token document length. Results are checkpointed to Drive after every sample.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import torch
IS_COLAB = 'google.colab' in sys.modules
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 2**30, 1))

In [ ]:
if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    uploaded = files.upload()
    archives = [Path('/content') / name for name in uploaded if name.startswith('boundary_interaction_kv_v12_stage_f2') and name.endswith('.zip')]
    WORK = Path('/content/bikv_v12_stage_f2_work')
    MIRROR = Path('/content/drive/MyDrive/bikv_runs')
else:
    WORK = Path.cwd() / 'bikv_v12_stage_f2_work'
    MIRROR = None
    archives = sorted(Path.cwd().glob('boundary_interaction_kv_v12_stage_f2*.zip'))
if not archives:
    raise FileNotFoundError('Upload boundary_interaction_kv_v12_stage_f2.zip')
SOURCE = WORK / 'source_v12_stage_f2'
if SOURCE.exists():
    shutil.rmtree(SOURCE)
SOURCE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[-1]) as archive:
    archive.extractall(SOURCE)
projects = list(SOURCE.rglob('pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError(f'Expected one project, found {len(projects)}')
PROJECT = projects[0].parent
print('Project:', PROJECT)

In [ ]:
os.environ['HF_HOME'] = '/content/hf_cache'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
if sys.version_info >= (3, 13):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'fire==0.7.1'], check=True)
print('Installed project and pinned dependencies.')

In [ ]:
base = json.loads((PROJECT / 'configs' / 'v12_stage_f2_medium.json').read_text())
base['data_path'] = str(WORK / 'data' / 'stage_f2_medium_seed_20261014.jsonl')
base['frozen_models_path'] = str(PROJECT / 'configs' / 'v12_stage_f2_frozen_models.json')
base['output_root'] = str(WORK / 'runs')
base['mirror_root'] = str(MIRROR) if MIRROR else None
runtime_path = WORK / 'runtime_v12_stage_f2.json'
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(base, indent=2), encoding='utf-8')
def stream(command):
    print(' '.join(map(str, command)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONPATH': f"{PROJECT / 'src'}:{PROJECT / 'scripts'}"}
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
stream([sys.executable, '-u', '-m', 'bikv.cli', 'prepare', '--config', str(runtime_path)])
print(runtime_path.read_text())

## Medium Stage F2
The first phase is a qualification gate. If it passes, the same cell continues with the frozen comparison. Re-run after a disconnect to resume from Drive-backed checkpoints.

In [ ]:
stream([sys.executable, '-u', str(PROJECT / 'scripts' / 'stage_f2_medium.py'), '--config', str(runtime_path)])

In [ ]:
run_root = WORK / 'runs' / base['run_id']
for name in ('qualification_gate.json', 'status.json'):
    path = run_root / name
    print(name, ':', path.read_text() if path.exists() else 'not written yet')
bundle = WORK / 'runs' / f"{base['run_id']}_artifacts.zip"
print('Bundle:', bundle, '| exists:', bundle.exists())
if IS_COLAB and bundle.exists():
    files.download(str(bundle))